# 提示された特徴量アイディアを `54_` に載せて検証（`73_`）

## まず: 提示されたアイディアの多くは既に検証済み

作業を始める前に、既存の記録と突き合わせた結果を出しておく。**同じ実験を繰り返さないため**である。

| 提示されたアイディア | 既存の実装/検証 | 結果 |
|---|---|---|
| Target Encoding（部署ID）| `15_`〜、KFold+リーク修正済み | **採用済み**（`dept_target_enc`・`dept_size`が441列に存在）|
| Target Encoding（上司ID）| EDA v6 の上司ランダム効果 | ❌ 観測分散は二項ノイズの1.12倍、LOO相関0.072 → 不採用 |
| 部署ID・上司IDの変更回数 | 441列の `catchange` グループ | **採用済み**。上司ID変更の生の効果量は 2.8pt/p=0.18（無信号）|
| **等級・役割の上昇フラグ** | EDA v3 分析1 | ⛔ **0-23ヶ月で等級・役割が変化した社員はゼロ**。原理的に作れない（本NB第2節で再確認）|
| 給与の差分・増加率 | 441列の月次集約 + `初任給_等級内偏差` 等 | **採用済み** |
| グループ集約（ブラック部署度・離職伝染）| EDA v6、`monthly-data-information-ceiling` | ❌ 離職伝染 r=0.002〜-0.024（完全null）、部署は「職種の再パッケージ」|
| 自己学習パース（総時間・トピック数）| `59_` SLブロック | ❌ 検証 **-0.0115**（過去最大級の改善）→ **Public +0.002273 悪化** |
| テキストの文字数 | 441列の `{col}_len`・`text_total_chars` | **採用済み** |
| キーワードフラグ（ポジ/ネガ）| EDA v4（手選び6語）・v6（12文字n-gram総当たり）| ❌ v4は6/6が p>0.15、v6は **Bonferroni後に有意ゼロ** |
| TF-IDF + SVD | 441列に15次元×3列 | **採用済み** |
| **BERT等の事前学習済み埋め込み** | `19_` | ⚠️ **e5-small(多言語)** で Public 0.564355 vs TF-IDF 0.550352（悪化）。**日本語特化BERTは未検証**（fugashi依存でColabが不安定になり断念した経緯あり）|
| 時系列トレンド（傾き）| ブロックN（モメンタム）| ❌ ローカルとColabで符号反転、不採用 |
| 初期と直近のギャップ | 441列に `残業時間_late_minus_early` 等が存在（重要度20位以内）| **採用済み** |
| 孤立・放置シグナル | `21_` ブロックH（情報共有ゼロ連続月数等）| ❌ Public 0.540507（悪化）|
| バーンアウト指標 | `52_` 残業dip-recovery | ❌ 19.5pt/p=2.5e-18 → Public +0.0019/+0.0005（両構成とも悪化）|

さらに **EDA v7** で、現行モデルの残差に対する部分集団スキャン（単変量94・交互作用2,110の計2,204検定）を
行い **Bonferroni通過ゼロ**。`monthly-data-information-ceiling` では
「アウトカム直前の窓を使ってすら月次データは定数予測から0.009しか改善しない」ことも確認済み。

**したがって本ノートブックの期待値は低い。** それでも作る理由は2つある。

1. **日本語特化BERTだけは本当に未検証**（`19_`は多言語e5-small。日本語BERTは着手して断念した）
2. **これまでの却下判定は分解能±0.0099の検証セットで下されていた**（第107節）。
   本NBは**分解能±0.0043の新しい検証設計**で測り直すので、過去の判定より信頼できる

---

## 本ノートブックの中核: 検証設計を入れ替える

第107節で確定した通り、535名の chronological 検証は **2構成の差の分解能が ±0.0099** しかなく、
**本プロジェクトが当ててきた改善8件すべてがその下**だった。この物差しで却下したブロックは、
「効かない」のではなく「測れなかった」だけかもしれない。

| 検証設計 | n | 2構成差の分解能 |
|---|---|---|
| 従来（入社日順80/20・生存者のみ）| 535 | ±0.0099 |
| **本NB（ランダム5-fold OOF・生存者のみで採点）** | **2,632** | **±0.0043** |

時系列分割を選んだ根拠（Train/Test の分布シフト）は EDA v7 で
**「シフトは給与1列に局在し、補正しても予測は動かない＝P(y|x)は安定」**と否定済み。
検証に必要なのは絶対値の予測ではなく順位付けであり、順位付けにはバイアスより分解能が効く。

**部署Target Encodingは fold ごとに再計算する**（fold の学習側IDだけで fit）。
これをしないと検証foldのラベルがTEに漏れ、新しい検証設計そのものが信用できなくなる。

## 事前登録

- ベースライン = `54_` の `R0_memofix_plus_LM`（444列、Public 0.515030）
- **各ブロックを1つずつ単体で追加**する。組み合わせ探索はしない（`39_`の多重比較の罠）
- 判定は **OOF(生存者2,632名)の差 vs 分解能±0.0043**、および Test予測のMAD vs ノイズ床0.02122
- **本NBでは提出ファイルを作らない**

In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00:00:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 15.6 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "73_feature_ideas_on_54"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-16 16:35:44] [INFO] === [73_feature_ideas_on_54] 実験開始 ===


INFO:73_feature_ideas_on_54:=== [73_feature_ideas_on_54] 実験開始 ===


[2026-08-16 16:35:45] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


INFO:73_feature_ideas_on_54:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


[2026-08-16 16:35:45] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/73_feature_ideas_on_54_checkpoint.csv


INFO:73_feature_ideas_on_54:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/73_feature_ideas_on_54_checkpoint.csv


[2026-08-16 16:35:45] [INFO] チェックポイントは未作成（新規実行）


INFO:73_feature_ideas_on_54:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-16 16:35:52] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:73_feature_ideas_on_54:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-16 16:35:52] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:73_feature_ideas_on_54:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-16 16:35:52] [INFO] 定着率: 0.5647


INFO:73_feature_ideas_on_54:定着率: 0.5647


[2026-08-16 16:35:52] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:73_feature_ideas_on_54:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-16 16:35:52] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:73_feature_ideas_on_54:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-16 16:35:52] [INFO] Test  早期退職者: 0名 / 2502名


INFO:73_feature_ideas_on_54:Test  早期退職者: 0名 / 2502名


[2026-08-16 16:35:52] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:73_feature_ideas_on_54:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-16 16:35:52] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:73_feature_ideas_on_54:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-16 16:35:53] [INFO] ------------------------------------------------------------


INFO:73_feature_ideas_on_54:------------------------------------------------------------


[2026-08-16 16:35:53] [INFO] split非依存の基本特徴量を生成中...


INFO:73_feature_ideas_on_54:split非依存の基本特徴量を生成中...


[2026-08-16 16:35:53] [INFO] ------------------------------------------------------------


INFO:73_feature_ideas_on_54:------------------------------------------------------------


[2026-08-16 16:44:24] [INFO] split非依存の基本特徴量生成完了


INFO:73_feature_ideas_on_54:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-16 16:44:25] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:73_feature_ideas_on_54:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-16 16:44:27] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:73_feature_ideas_on_54:入社時メモ: SVD累積寄与率=0.760


[2026-08-16 16:44:33] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:73_feature_ideas_on_54:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-16 16:44:35] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:73_feature_ideas_on_54:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-16 16:44:35] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:73_feature_ideas_on_54:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-16 16:44:36] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:73_feature_ideas_on_54:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-16 16:47:46] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:73_feature_ideas_on_54:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-16 16:47:46] [INFO] Persona単位の基本特徴量を生成中...


INFO:73_feature_ideas_on_54:Persona単位の基本特徴量を生成中...


[2026-08-16 16:47:46] [INFO] Persona単位の基本特徴量処理完了


INFO:73_feature_ideas_on_54:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、49_でパーサーを修正）

`extract_workstyle_section()` に、見出し（`勤務地・働き方：`）が無い書式Bのフォールバックを追加した。
それ以外（`classify_reloc` / `extract_desired_location_v1` / `v2`）は `40_` と同一。


In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 49_: 見出しがない書式B（276件、5.24%）のフォールバック。
    # 「勤務地・転居・在宅勤務」に言及する行を拾い、疑似セクションとして返す。
    # 以降のclassify_reloc/extract_desired_location_v1/v2はre.searchで探すだけなので、
    # 複数行を連結してもそのまま動く。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


def _report_ws_coverage_fix(train_persona, test_persona):
    """49_の修正がどれだけカバー率を回復させたかをログに残す（診断専用、学習には影響しない）"""
    def old_fn(text):
        if pd.isna(text):
            return None
        m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
        return m.group(1).strip() if m else None

    all_persona = pd.concat([train_persona[["入社時メモ"]], test_persona[["入社時メモ"]]], ignore_index=True)
    ws_old = all_persona["入社時メモ"].apply(old_fn)
    ws_new = all_persona["入社時メモ"].apply(extract_workstyle_section)
    n = len(all_persona)
    logger.info(f"[49_診断] 見出し欠落 修正前 {ws_old.isna().sum()}件({ws_old.isna().sum()/n:.2%}) "
                f"→ 修正後 {ws_new.isna().sum()}件({ws_new.isna().sum()/n:.2%})")


_report_ws_coverage_fix(train_persona, test_persona)


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())


[2026-08-16 16:47:47] [INFO] [49_診断] 見出し欠落 修正前 276件(5.24%) → 修正後 2件(0.04%)


INFO:73_feature_ideas_on_54:[49_診断] 見出し欠落 修正前 276件(5.24%) → 修正後 2件(0.04%)


[2026-08-16 16:47:47] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:73_feature_ideas_on_54:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-16 16:47:47] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:73_feature_ideas_on_54:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-16 16:47:47] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:73_feature_ideas_on_54:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2397
1     364
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2385
1     376
Name: count, dtype: int64


## 5b. L2×M交互作用フラグ（`54_`で追加、Job Embeddedness理論由来）


In [14]:
# ============================================================
# 54_: L2×Mのリスク要因数（Job Embeddedness理論: 複数の埋め込み不足の重なり）
#   ブロックM（専攻職種の分析的ミスマッチ）は29_で単体却下済み（GBDT redundancy）。
#   L2（転居x勤務地_状態_v2 = "非許容_不一致"）との組み合わせをフラグ化する。
#
#   ローカルEDAでの生の効果量（Cochran-Armitage傾向検定, p≈0・機械精度限界）:
#     リスク要因0個(n=1885): 定着率67.0%
#     リスク要因1個(n=653) : 定着率28.8%
#     リスク要因2個(n=47)  : 定着率 2.1%
#   単なるAND(2個該当)だけでなく、0→1→2ときれいな段階的用量反応があったため、
#   二値フラグに加えて順序尺度の"risk_count"も特徴量として渡す。
#   ただし該当セルのp値は採用基準(p<1e-20)を厳密には満たさないセルもあり、
#   単一の事前登録済み検証として扱う（閾値をチューニングしない）。
# ============================================================

_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)

    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)

    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad  # 0/1/2の順序尺度（用量反応をそのまま渡す）

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info(f"L2xMリスク特徴量: Train {train_l2m.shape}, Test {test_l2m.shape}")
print(train_l2m["L2xM_リスク要因数"].value_counts().sort_index())


[2026-08-16 16:47:47] [INFO] L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


INFO:73_feature_ideas_on_54:L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


L2xM_リスク要因数
0    2025
1     689
2      47
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [15]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        tf = tf.merge(train_l2m, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_l2m, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


---
## 9. ベースラインの組み立て（`54_` と同一の444列）

In [16]:
import time                      # 54_ の import セルには入っていないのでここで追加
from sklearn.metrics import log_loss

BLOCK = {"L2"}    # 54_ と同一（L_v2 + L2xM が入る）

logger.info("[全件] 特徴量を組み立て中...")
ag_full, _empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)
assert len(_empty) == 0

def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]

BASE_FEATS = _feature_cols(ag_full)
y_all = ag_full[TARGET_COL].values.astype(int)
print(f"ベースライン {len(BASE_FEATS)} 列 / 学習 {len(ag_full)}名 / Test {len(test_features_full)}名")

# 生存者マスク（採点対象）。Test には早期退職者が0名なので、検証も生存者だけで採点する
IS_SURV = ~ag_full.index.isin(EARLY_LEAVER_IDS)
print(f"生存者 {IS_SURV.sum()}名 / 早期退職者 {(~IS_SURV).sum()}名（学習には使うが採点からは外す）")


[2026-08-16 16:47:48] [INFO] [全件] 特徴量を組み立て中...


INFO:73_feature_ideas_on_54:[全件] 特徴量を組み立て中...


ベースライン 444 列 / 学習 2761名 / Test 2502名
生存者 2632名 / 早期退職者 129名（学習には使うが採点からは外す）


## 10. 【再確認】等級・役割の上昇は原理的に作れない

提示されたアイディアのうち「等級・役割の上昇フラグ」は、EDA v3 で
「0-23ヶ月以内に等級・役割が変化した社員はゼロ」と判定済み。実データで再確認する。

In [17]:
_chk = train_monthly.sort_values([ID_COL, "経過月数"]).groupby(ID_COL).agg(
    等級_初=("等級", "first"), 等級_終=("等級", "last"),
    役割_初=("役割", "first"), 役割_終=("役割", "last"))
n_grade = (_chk["等級_初"] != _chk["等級_終"]).sum()
n_role  = (_chk["役割_初"] != _chk["役割_終"]).sum()
print(f"0-23ヶ月で等級が変化した社員: {n_grade}名 / 役割が変化: {n_role}名（Train {len(_chk)}名中）")
_chk2 = test_monthly.sort_values([ID_COL, "経過月数"]).groupby(ID_COL).agg(
    等級_初=("等級", "first"), 等級_終=("等級", "last"))
print(f"Test で等級が変化した社員: {(_chk2['等級_初'] != _chk2['等級_終']).sum()}名 / {len(_chk2)}名")
print("→ 変化がゼロなら『昇進フラグ』は定数列にしかならない。本NBでは作らない。")


0-23ヶ月で等級が変化した社員: 0名 / 役割が変化: 0名（Train 2761名中）
Test で等級が変化した社員: 0名 / 2502名
→ 変化がゼロなら『昇進フラグ』は定数列にしかならない。本NBでは作らない。


## 11. ブロックの実装

提示されたアイディアを**1ブロック1関数**で実装する。既に却下済みのものも、
新しい検証設計（分解能±0.0043）で測り直す価値があるので**実装して回す**。

In [18]:
from scipy import stats as _st

def _slope(v):
    v = np.asarray(v, dtype=float); ok = ~np.isnan(v)
    if ok.sum() < 2: return 0.0
    return np.polyfit(np.arange(len(v))[ok], v[ok], 1)[0]

def _maxrun_zero(v):
    best = cur = 0
    for x in v:
        cur = cur + 1 if (x == 0 or pd.isna(x)) else 0
        best = max(best, cur)
    return best

TREND_COLS = ["残業時間", "情報共有件数", "在宅勤務日数", "有給取得日数",
              "360度評価_主体度", "360度評価_信頼度", "顧客満足度評価"]

# ---------- TR: 時系列トレンド（傾き）＋ 初期と直近のギャップ ----------
def blk_TR(monthly, ids):
    out = []
    for eid, g in monthly.sort_values([ID_COL, "経過月数"]).groupby(ID_COL, sort=False):
        r = {ID_COL: eid}
        for c in TREND_COLS:
            v = g[c].to_numpy(dtype=float)
            r[f"TR_{c}_傾き"] = _slope(v)
            早 = g[g["経過月数"] <= 2][c].mean(); 直 = g[g["経過月数"] >= 21][c].mean()
            r[f"TR_{c}_初期直近差"] = 直 - 早
        out.append(r)
    return pd.DataFrame(out)

# ---------- ZS: 孤立・放置シグナル ----------
def blk_ZS(monthly, ids):
    out = []
    for eid, g in monthly.sort_values([ID_COL, "経過月数"]).groupby(ID_COL, sort=False):
        n = len(g)
        out.append({ID_COL: eid,
            "ZS_情報共有ゼロ最長": _maxrun_zero(g["情報共有件数"].to_numpy()),
            "ZS_情報共有ゼロ率": float((g["情報共有件数"] == 0).mean()),
            "ZS_面談ゼロ率": float((g["上司との面談実施回数"] == 0).mean()),
            "ZS_面談ゼロ最長": _maxrun_zero(g["上司との面談実施回数"].to_numpy()),
            "ZS_評価未更新最長": _maxrun_zero(g["360度評価更新フラグ"].to_numpy()),
            "ZS_評価更新率": float(g["360度評価更新フラグ"].mean())})
    return pd.DataFrame(out)

# ---------- BO: バーンアウト（残業高止まり × 欠勤増加） ----------
def blk_BO(monthly, ids):
    ot_hi = monthly["残業時間"].quantile(0.75)
    out = []
    for eid, g in monthly.sort_values([ID_COL, "経過月数"]).groupby(ID_COL, sort=False):
        ot = g["残業時間"].to_numpy(dtype=float); ab = g["欠勤日数"].to_numpy(dtype=float)
        # 直近6ヶ月で残業が上位25%水準を維持し、かつ欠勤の傾きが正
        late = slice(max(0, len(g) - 6), len(g))
        out.append({ID_COL: eid,
            "BO_残業高止まり月数": int((ot >= ot_hi).sum()),
            "BO_欠勤傾き": _slope(ab),
            "BO_フラグ": int((np.nanmean(ot[late]) >= ot_hi) and (_slope(ab) > 0))})
    return pd.DataFrame(out)

# ---------- SL: 自己学習パース（59_ で不採用、再測定） ----------
def blk_SL(monthly, ids):
    def hours(t):
        if pd.isna(t) or t == "受講なし": return 0.0
        return sum(float(x) for x in re.findall(r"([\d.]+)時間", str(t)))
    def topics(t):
        if pd.isna(t) or t == "受講なし": return []
        return [n.strip().replace(" ", "") for n in re.findall(r"([^｜：]+)：[\d.]+時間", str(t))]
    m = monthly.copy()
    m["_h"] = m["自己学習（詳細）"].apply(hours)
    m["_t"] = m["自己学習（詳細）"].apply(topics)
    out = []
    for eid, g in m.sort_values([ID_COL, "経過月数"]).groupby(ID_COL, sort=False):
        th = [t for lst in g["_t"] for t in lst]
        out.append({ID_COL: eid,
            "SL_総学習時間": float(g["_h"].sum()), "SL_学習月数": int((g["_h"] > 0).sum()),
            "SL_平均学習時間": float(g["_h"].mean()), "SL_学習時間傾き": _slope(g["_h"].to_numpy()),
            "SL_ユニークテーマ数": len(set(th)), "SL_延べテーマ数": len(th)})
    return pd.DataFrame(out)

# ---------- GA: グループ集約（上司・部署の環境スコア） ----------
def blk_GA(monthly, ids):
    '''同じ上司/部署に紐づく「他人」の平均で環境を測る（自分は必ず除外＝leave-one-out）。
       ラベルは一切使わない（離職率は使わない）ので、TEのようなリークは起きない。'''
    m0 = monthly[monthly["経過月数"] == 0]
    peer = monthly.groupby(ID_COL).agg(_ot=("残業時間", "mean"), _pto=("有給取得日数", "mean"),
                                       _ab=("欠勤日数", "mean")).reset_index()
    base = m0[[ID_COL, "上司ID", "部署ID"]].merge(peer, on=ID_COL, how="left")
    out = base[[ID_COL]].copy()
    for key, tag in [("上司ID", "上司"), ("部署ID", "部署")]:
        for col, nm in [("_ot", "残業"), ("_pto", "有給"), ("_ab", "欠勤")]:
            g = base.groupby(key)[col]
            s, n = g.transform("sum"), g.transform("count")
            out[f"GA_{tag}_他者平均_{nm}"] = ((s - base[col]) / (n - 1)).to_numpy()   # 自分を除く
        out[f"GA_{tag}_人数"] = base.groupby(key)[col].transform("count").to_numpy()
    return out

print("ブロック関数を定義した: TR / ZS / BO / SL / GA")


ブロック関数を定義した: TR / ZS / BO / SL / GA


In [19]:
# ---------- KW: キーワードフラグ（v4/v6 で不採用、再測定） ----------
NEG_WORDS = ["遅刻", "課題", "不足", "苦手", "難しい", "抱え込", "確認漏れ", "手戻り",
             "受け身", "指示待ち", "戸惑", "偏り", "限定的"]
POS_WORDS = ["期待", "優秀", "リーダー", "主体的", "自律", "率先", "丁寧", "着実",
             "信頼", "貢献", "柔軟", "前向き", "素地"]
KW_TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

def blk_KW(persona):
    out = pd.DataFrame({ID_COL: persona[ID_COL].values})
    for col in KW_TEXT_COLS:
        t = persona[col].fillna("").astype(str)
        tag = {"入社時メモ": "memo", "上司からのフィードバック": "sup", "同僚からのフィードバック": "peer"}[col]
        out[f"KW_{tag}_neg数"] = sum(t.str.contains(w, regex=False).astype(int) for w in NEG_WORDS).to_numpy()
        out[f"KW_{tag}_pos数"] = sum(t.str.contains(w, regex=False).astype(int) for w in POS_WORDS).to_numpy()
        out[f"KW_{tag}_極性差"] = out[f"KW_{tag}_pos数"] - out[f"KW_{tag}_neg数"]
    return out

train_KW, test_KW = blk_KW(train_persona), blk_KW(test_persona)
print(f"KW: {train_KW.shape[1]-1}列")
print(train_KW.filter(like="極性差").describe().round(2).to_string())


KW: 9列
       KW_memo_極性差  KW_sup_極性差  KW_peer_極性差
count      2761.00     2761.00      2761.00
mean          1.48       -0.16         0.05
std           0.87        0.65         0.46
min          -1.00       -3.00        -2.00
25%           1.00        0.00         0.00
50%           2.00        0.00         0.00
75%           2.00        0.00         0.00
max           4.00        2.00         2.00


## 12. ブロック BE: 日本語特化BERT の埋め込み（**本NBで唯一の真の新規**）

`19_` は多言語モデル `intfloat/multilingual-e5-small` を使い、Public 0.564355（TF-IDFの0.550352より悪化）で
不採用になった。当初予定していた**日本語特化モデルは fugashi 依存でColabが不安定になり断念**しており、
**実際には一度も試されていない**。

CPUでも動くよう、`cl-tohoku/bert-base-japanese-v3`（fugashi必須）を第一候補に、
失敗したら分かち書き不要の代替へ自動フォールバックする。

- `[CLS]` トークンのベクトル（768次元）を文章表現として使う
- PCA は **Train+Test を結合してから fit**（ラベルを使わないのでリークにならない）
- 次元は `19_` と揃えて **15次元**（`19_`は384→PCA15。ここは探索しない）

> ⚠️ `BE_ENABLE = False` にすればスキップできる。CPUで5,263件×3列の推論は10〜30分かかる。

In [20]:
BE_ENABLE = True          # Falseでスキップ
BE_DIM    = 15            # 19_ と同じ。探索しない
BE_COLS   = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

train_BE = test_BE = None
if BE_ENABLE:
    try:
        import subprocess, sys as _sys
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q",
                        "transformers", "fugashi", "unidic-lite", "torch"], check=False)
        import torch
        from transformers import AutoTokenizer, AutoModel
        from sklearn.decomposition import PCA

        MODEL_CANDIDATES = ["cl-tohoku/bert-base-japanese-v3",
                            "intfloat/multilingual-e5-small"]   # フォールバック
        tok = mdl = None
        for name in MODEL_CANDIDATES:
            try:
                tok = AutoTokenizer.from_pretrained(name)
                mdl = AutoModel.from_pretrained(name).eval()
                BE_MODEL = name
                print(f"✅ 埋め込みモデル: {name}")
                break
            except Exception as e:
                print(f"  {name} をロードできない: {type(e).__name__}: {str(e)[:100]}")
        if mdl is None:
            raise RuntimeError("埋め込みモデルを一つもロードできなかった")

        dev = "cuda" if torch.cuda.is_available() else "cpu"
        mdl.to(dev); print(f"  device={dev}")

        def embed(texts, bs=32):
            vs = []
            for i in range(0, len(texts), bs):
                b = [str(x) if pd.notna(x) and str(x).strip() else "記載なし" for x in texts[i:i+bs]]
                enc = tok(b, return_tensors="pt", truncation=True, max_length=128,
                          padding="max_length")
                enc = {k: v.to(dev) for k, v in enc.items()}
                with torch.no_grad():
                    o = mdl(**enc)
                vs.append(o.last_hidden_state[:, 0, :].cpu().numpy())   # [CLS]
            return np.vstack(vs)

        tr_parts, te_parts = [], []
        for col in BE_COLS:
            t0 = time.time()
            allv = embed(list(train_persona[col].values) + list(test_persona[col].values))
            p = PCA(n_components=BE_DIM, random_state=SEED).fit(allv)     # Train+Testでfit（ラベル不使用）
            z = p.transform(allv)
            tag = {"入社時メモ": "memo", "上司からのフィードバック": "sup", "同僚からのフィードバック": "peer"}[col]
            cols = [f"BE_{tag}_{i}" for i in range(BE_DIM)]
            tr_parts.append(pd.DataFrame(z[:len(train_persona)], columns=cols))
            te_parts.append(pd.DataFrame(z[len(train_persona):], columns=cols))
            print(f"  {col}: 累積寄与率={p.explained_variance_ratio_.sum():.3f} ({time.time()-t0:.0f}秒)")

        train_BE = pd.concat([pd.DataFrame({ID_COL: train_persona[ID_COL].values})] + tr_parts, axis=1)
        test_BE  = pd.concat([pd.DataFrame({ID_COL: test_persona[ID_COL].values})] + te_parts, axis=1)
        print(f"BE: {train_BE.shape[1]-1}列")
    except Exception as e:
        print(f"⚠️ BEブロックを作れなかったのでスキップする: {type(e).__name__}: {e}")
        train_BE = test_BE = None


config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  447MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 埋め込みモデル: cl-tohoku/bert-base-japanese-v3
  device=cpu
  入社時メモ: 累積寄与率=0.593 (637秒)
  上司からのフィードバック: 累積寄与率=0.498 (638秒)
  同僚からのフィードバック: 累積寄与率=0.500 (634秒)
BE: 45列


In [21]:
# ---- 月次由来ブロックの生成（重いのでここで一括） ----
BLOCKS = {}
for name, fn in [("TR", blk_TR), ("ZS", blk_ZS), ("BO", blk_BO), ("SL", blk_SL), ("GA", blk_GA)]:
    t0 = time.time()
    tr = fn(train_monthly, train_ids); te = fn(test_monthly, test_ids)
    BLOCKS[name] = (tr, te)
    print(f"  {name}: {tr.shape[1]-1}列  ({time.time()-t0:.0f}秒)")
BLOCKS["KW"] = (train_KW, test_KW)
if train_BE is not None:
    BLOCKS["BE"] = (train_BE, test_BE)
print(f"\n生成したブロック: {list(BLOCKS)}")


  TR: 14列  (34秒)
  ZS: 6列  (2秒)
  BO: 3列  (2秒)
  SL: 6列  (3秒)
  GA: 8列  (0秒)

生成したブロック: ['TR', 'ZS', 'BO', 'SL', 'GA', 'KW', 'BE']


## 13. 新しい検証設計（分解能 ±0.0043）

- 全2,761名の **ランダム5-fold StratifiedKFold OOF**
- 採点は **生存者2,632名のみ**（Test に早期退職者が0名のため）
- **部署TEは fold ごとに再計算**（fold の学習側IDだけで fit）。しないと検証foldのラベルが漏れる
- ハイパーパラメータは `54_` の `A_PARAMS` 固定・反復560固定・3シード平均

In [22]:
from sklearn.model_selection import StratifiedKFold

A_PARAMS = {"depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
            "border_count": 218, "bagging_temperature": 0.6787467566574921,
            "random_strength": 1.438494697238285}
ITER_FIXED = 560
SEEDS_73 = [42, 2024, 7]
N_FOLDS = 5
TE_COLS = ["dept_target_enc", "dept_size"]

_all_ids = ag_full.index.to_numpy()

def _te_for_fold(fit_ids):
    '''fold の学習側IDだけで部署TEを作り直す（リーク防止）。'''
    tr_te, te_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=set(fit_ids), seed=SEED, n_splits=5, smoothing=10)
    return tr_te.set_index(ID_COL), te_te.set_index(ID_COL)


def run_config(label, extra_names):
    X = ag_full[BASE_FEATS].copy(); Xte = test_features_full[BASE_FEATS].copy()
    for nm in extra_names:
        tr, te = BLOCKS[nm]
        X = X.join(tr.set_index(ID_COL), how="left")
        Xte = Xte.join(te.set_index(ID_COL), how="left")
    feats = list(X.columns)
    obj = [c for c in feats if X[c].dtype == "object"]
    X[obj] = X[obj].astype(str); Xte[obj] = Xte[obj].astype(str)
    num = [c for c in feats if c not in obj]
    X[num] = X[num].fillna(-999); Xte[num] = Xte[num].fillna(-999)

    oof = np.zeros(len(X)); tps = []
    for s in SEEDS_73:
        o = np.zeros(len(X))
        for tri, vai in StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y_all):
            Xf = X.copy()
            tr_te, _ = _te_for_fold(_all_ids[tri])          # ★ foldごとにTEを作り直す
            for c in TE_COLS:
                if c in Xf.columns: Xf[c] = tr_te.loc[Xf.index, c].to_numpy()
            m = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER_FIXED, random_seed=s,
                                      verbose=False, cat_features=obj, task_type="CPU")
            m.fit(Xf.iloc[tri], y_all[tri])
            o[vai] = m.predict_proba(Xf.iloc[vai])[:, 1]
        oof += o / len(SEEDS_73)
        m2 = cb.CatBoostClassifier(**A_PARAMS, iterations=int(ITER_FIXED*1.25), random_seed=s,
                                   verbose=False, cat_features=obj, task_type="CPU")
        m2.fit(X, y_all); tps.append(m2.predict_proba(Xte)[:, 1])
    tp = np.mean(tps, 0)
    return dict(config=label, n_features=len(feats),
                oof_surv=log_loss(y_all[IS_SURV], oof[IS_SURV]),
                oof_all=log_loss(y_all, oof)), oof, tp


# 事前登録した構成（ベースライン + 各ブロック単体。組み合わせはしない）
CONFIGS = [("baseline", [])] + [(f"+{k}", [k]) for k in BLOCKS]
print(f"実行する構成 {len(CONFIGS)}件: {[c[0] for c in CONFIGS]}")


実行する構成 8件: ['baseline', '+TR', '+ZS', '+BO', '+SL', '+GA', '+KW', '+BE']


In [23]:
results, oofs, tests = [], {}, {}
for label, ex in CONFIGS:
    t0 = time.time()
    r, o, t = run_config(label, ex)
    results.append(r); oofs[label] = o; tests[label] = t
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{label.replace('+','plus_')}_oof.npy", o)
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{label.replace('+','plus_')}_testpreds.npy", t)
    print(f"  {label:12s} 列{r['n_features']:4d}  OOF(生存者) {r['oof_surv']:.6f}  "
          f"OOF(全体) {r['oof_all']:.6f}  ({time.time()-t0:.0f}秒)")
    logger.info(str(r))

R = pd.DataFrame(results)
base = R.loc[R.config == "baseline", "oof_surv"].iloc[0]
R["対baseline"] = R["oof_surv"] - base
R["Test_MAD"] = [np.abs(tests[c] - tests["baseline"]).mean() for c in R.config]
R.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)
print(); print(R.round(6).to_string(index=False))


  baseline     列 444  OOF(生存者) 0.513862  OOF(全体) 0.495081  (127秒)
[2026-08-16 17:23:16] [INFO] {'config': 'baseline', 'n_features': 444, 'oof_surv': 0.5138624219721779, 'oof_all': 0.4950806951146861}


INFO:73_feature_ideas_on_54:{'config': 'baseline', 'n_features': 444, 'oof_surv': 0.5138624219721779, 'oof_all': 0.4950806951146861}


  +TR          列 458  OOF(生存者) 0.513227  OOF(全体) 0.494128  (138秒)
[2026-08-16 17:25:33] [INFO] {'config': '+TR', 'n_features': 458, 'oof_surv': 0.5132271724278521, 'oof_all': 0.4941284134724301}


INFO:73_feature_ideas_on_54:{'config': '+TR', 'n_features': 458, 'oof_surv': 0.5132271724278521, 'oof_all': 0.4941284134724301}


  +ZS          列 450  OOF(生存者) 0.515136  OOF(全体) 0.496264  (127秒)
[2026-08-16 17:27:40] [INFO] {'config': '+ZS', 'n_features': 450, 'oof_surv': 0.5151362176985962, 'oof_all': 0.49626358878342974}


...

  +KW          列 453  OOF(生存者) 0.514889  OOF(全体) 0.496190  (126秒)
[2026-08-16 17:36:16] [INFO] {'config': '+KW', 'n_features': 453, 'oof_surv': 0.5148888242412089, 'oof_all': 0.49619016508565505}


INFO:73_feature_ideas_on_54:{'config': '+KW', 'n_features': 453, 'oof_surv': 0.5148888242412089, 'oof_all': 0.49619016508565505}


  +BE          列 489  OOF(生存者) 0.514919  OOF(全体) 0.496231  (133秒)
[2026-08-16 17:38:29] [INFO] {'config': '+BE', 'n_features': 489, 'oof_surv': 0.5149193160646953, 'oof_all': 0.4962309233583923}


INFO:73_feature_ideas_on_54:{'config': '+BE', 'n_features': 489, 'oof_surv': 0.5149193160646953, 'oof_all': 0.4962309233583923}



  config  n_features  oof_surv  oof_all  対baseline  Test_MAD
baseline         444  0.513862 0.495081   0.000000  0.000000
     +TR         458  0.513227 0.494128  -0.000635  0.023194
     +ZS         450  0.515136 0.496264   0.001274  0.022358
     +BO         447  0.513782 0.494981  -0.000081  0.021936
     +SL         450  0.506533 0.488063  -0.007329  0.037608
     +GA         452  0.514746 0.496058   0.000883  0.023581
     +KW         453  0.514889 0.496190   0.001026  0.022738
     +BE         489  0.514919 0.496231   0.001057  0.031507


## 14. 判定

分解能を**実測**してから読む。事前に決めた通り、`対baseline` が分解能を超えないブロックは
「効果なし」ではなく **「測れなかった」** と記録する。

In [24]:
# ---- 分解能の実測（対応ありブートストラップ、生存者2,632名） ----
rng = np.random.RandomState(0)
ys, ob = y_all[IS_SURV], oofs["baseline"][IS_SURV]
# 分解能は「baseline vs 各ブロック」の対応ありブートストラップで、構成ごとに測る
print(f"=== 判定（OOF生存者 {IS_SURV.sum()}名、ノイズ床 MAD 0.02122）===\n")
print(f'{"構成":12s} {"OOF(生存者)":>11s} {"対base":>10s} {"分解能±":>9s} {"判定":>10s} {"Test_MAD":>9s}')
for _, row in R.iterrows():
    if row.config == "baseline":
        print(f'{row.config:12s} {row.oof_surv:11.6f} {"—":>10s} {"—":>9s} {"—":>10s} {"—":>9s}')
        continue
    oa = oofs[row.config][IS_SURV]
    d = []
    for _ in range(2000):
        i = rng.randint(0, len(ys), len(ys))
        d.append(log_loss(ys[i], oa[i], labels=[0,1]) - log_loss(ys[i], ob[i], labels=[0,1]))
    half = 1.96*np.std(d)
    verdict = ("改善" if row.対baseline < -half else "悪化" if row.対baseline > half else "測定不能")
    print(f'{row.config:12s} {row.oof_surv:11.6f} {row.対baseline:+10.6f} {half:9.5f} {verdict:>10s} {row.Test_MAD:9.5f}')
print("\n  『測定不能』= 差が分解能未満。効果がないのではなく、この設計では判定できないという意味。")
print("  Test_MAD がノイズ床0.02122を超えていれば、予測としては別モデルになっている。")


=== 判定（OOF生存者 2632名、ノイズ床 MAD 0.02122）===

構成              OOF(生存者)      対base      分解能±         判定  Test_MAD
baseline        0.513862          —         —          —         —
+TR             0.513227  -0.000635   0.00269       測定不能   0.02319
+ZS             0.515136  +0.001274   0.00265       測定不能   0.02236
+BO             0.513782  -0.000081   0.00266       測定不能   0.02194
+SL             0.506533  -0.007329   0.00409         改善   0.03761
+GA             0.514746  +0.000883   0.00274       測定不能   0.02358
+KW             0.514889  +0.001026   0.00261       測定不能   0.02274
+BE             0.514919  +0.001057   0.00347       測定不能   0.03151

  『測定不能』= 差が分解能未満。効果がないのではなく、この設計では判定できないという意味。
  Test_MAD がノイズ床0.02122を超えていれば、予測としては別モデルになっている。


## 15. 読み方と次のアクション

### 事前登録した運用

- **`改善` と出たブロックだけ**を Public 提出の候補にする。`測定不能` は候補にしない
- 分解能±0.0043前後なので、**±0.004を超える改善が出れば過去の判定より信頼できる**
- ただし [[validation-asymmetry]] の通り「改善」方向は最終的に Public でしか確定しない。
  本NBの結果は**候補の絞り込み**であって確定ではない

### 期待値について（正直に）

冒頭の表の通り、BE 以外の全ブロックは既に別の形で不採用になっている。
EDA v7 の残差スキャン（2,204検定でBonferroni通過ゼロ）と
[[monthly-data-information-ceiling]]（月次データの情報の天井）を踏まえると、
**全ブロックが `測定不能` に終わる可能性が高い**。

それでも意味があるのは:
1. **BE（日本語BERT）は本当に未検証**であること
2. 過去の却下が**分解能±0.0099の物差し**で下されていたのに対し、
   ここでは**±0.0043**で測り直していること。もし過去の判定が誤りだったなら、ここで浮く

### やらないこと

- **提出ファイルを作らない**（候補が出たら別途）
- **組み合わせ探索をしない**（`39_` の多重比較の罠）
- ハイパーパラメータを触らない（[[hyperparameter-retuning-exhausted]]）